[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# The Index


## What you will be able to do

Choose the labels that identify a table's rows, look rows up by them, move them back into columns,
and label each reading by its station and its hour together.

## The idea

### The problem

The readings arrived with an index nobody chose, the numbers 0 to 35039, one per row. Those numbers
say where a row sits and nothing about what it is. Filter the table to Oslo and the labels become 1,
5, 9 and so on, which are still row numbers, only now with gaps. To find Oslo's reading at 15:00 on
17 July you write two conditions, one on the station and one on the time, every time you ask.

The table of stations has the same problem in a smaller form. "What is Tromso's latitude?" is a
lookup, and with row numbers as labels it becomes a filter on the `station` column followed by
picking the one value out of the result. A dictionary would answer it in one step,
`latitudes["Tromso"]`, because its keys are the names you ask by.

Labels matter for more than lookups. The **Why Pandas** notebook lined two Series up by station
name, and that worked because both had the names as their index. A table whose index is row numbers
lines up with another by row number, which is rarely what you mean.

### What the index is

> The **index** of a DataFrame or Series holds the labels for its rows. `loc` looks rows up by
> those labels, operations between two tables line up their rows by them, and `set_index` and
> `reset_index` move a column into the index and back out. A **MultiIndex** labels each row with
> several values at once, such as a station and a time.

### Why it works that way

The index is kept apart from the columns. `set_index("station")` takes the `station` column out of
the table and makes it the labels, and `reset_index` puts the labels back as a column and numbers
the rows from 0 again. Nothing is lost either way, which is why moving a column in and out of the
index is an ordinary step in an analysis.

An index does not have to be unique. The stations table has one row per name, so `loc["Tromso"]`
returns one row. The readings have 8,760 rows per station, so the same lookup on a table indexed by
station returns all 8,760 of them. pandas allows both, and what comes back depends on the data,
which is the source of one of the errors in Common errors.

A MultiIndex has levels, here `station` and `time`, and a row is found by a value for each, written
as a tuple. pandas finds a single row in a MultiIndex in any order, and a range of rows only once
the index is sorted, because a range means "from here to there" and that needs an order. On a time
index, a date written as text, such as `"2025-07-17"`, selects every hour of that day.

### Where this shows up

A `groupby` result has the group keys as its index unless it is given `as_index=False`, which is why
the means in the **Why Pandas** notebook named their stations. The **Reshaping** notebook spreads an
index level across the columns with `unstack`, the **Merging** notebook turns a groupby's index back
into a column with `reset_index` so that it can join on it, and the **Dates and Times** notebook
resamples a series by its time index. The **loc and iloc** notebook compares looking rows up by
label with looking them up by position.

### What this notebook covers

- the index pandas gives a table, and what filtering does to it
- `set_index` on the stations, for lookups by name
- a time index, with a single hour, a range of hours and a whole day
- `reset_index`, with and without keeping the old labels
- a MultiIndex of station and time, sorted, sliced and cut across with `xs`
- five errors, among them a range that needs a sorted index and a lookup that returns 8,760 rows

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

stations = pd.DataFrame({"station": ["Bergen", "Oslo", "Tromso"], "latitude": [60.39, 59.91, 69.65]})
by_name = stations.set_index("station")
print(by_name)
print(by_name.loc["Tromso", "latitude"])
```

```
         latitude
station
Bergen      60.39
Oslo        59.91
Tromso      69.65
69.65
```

`set_index("station")` turned the names into the row labels, which is why `station` now sits on a
line of its own above them. `loc["Tromso", "latitude"]` looked up a row label and a column name, the
way you would look up a key in a dictionary.

## Setup

One library and three helpers for building the readings.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings

`readings` is the table of the four stations' hourly readings for 2025, 35,040 rows with the columns
`station`, `time` and `celsius`, and Svalbard's 24 readings for 2 March are `NaN`. `stations` holds
each station's latitude, with Kirkenes, which has no readings, as its last row.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### The index pandas gave the table

A table built without an index gets a `RangeIndex`, the numbers from 0 up, stored as a start, a stop
and a step instead of 35,040 separate values:

In [2]:
print(readings.index)

oslo_rows = readings[readings["station"] == "Oslo"]
print(oslo_rows.index[:5])

RangeIndex(start=0, stop=35040, step=1)
Index([1, 5, 9, 13, 17], dtype='int64')


The filtered table kept the labels of the rows it kept, 1, 5, 9 and onward, because Oslo is every
fourth row. Its index is now an ordinary `Index` of whole numbers, with gaps.

### Station names as the index

`set_index` makes a column the index:

In [3]:
by_name = stations.set_index("station")
by_name

,latitude
station,
Bergen,60.39
Oslo,59.91
Svalbard,78.22
Tromso,69.65
Kirkenes,69.73


`station` is no longer a column. It is the index, and its name is shown above the labels. `loc`
looks a row up by label, and a label and a column name together give one value:

In [4]:
print(by_name.loc["Tromso"])
print()
print("Svalbard's latitude:", by_name.loc["Svalbard", "latitude"])
print("one row per name:", by_name.index.is_unique)

latitude    69.65
Name: Tromso, dtype: float64

Svalbard's latitude: 78.22
one row per name: True


A single label returned that row as a Series, whose index is the table's column names. `is_unique`
says every label appears once, so every lookup by name returns one row.

### A time index

Oslo's readings with their times as the index make a Series that can be asked by date and time:

In [5]:
oslo = readings[readings["station"] == "Oslo"].set_index("time")["celsius"]

print(oslo.head(3))
print()
print("15:00 on 17 July:", oslo.loc["2025-07-17 15:00"])

time
2025-01-01 00:00:00   -3.5
2025-01-01 01:00:00   -5.3
2025-01-01 02:00:00   -5.3
Name: celsius, dtype: float64

15:00 on 17 July: 19.3


`set_index("time")` made the times the labels, and `["celsius"]` kept the one column. The text
`"2025-07-17 15:00"` was read as a time and matched the label. A slice between two times includes
both ends, because `loc` slices by label:

In [6]:
oslo.loc["2025-07-17 13:00":"2025-07-17 16:00"]

time
2025-07-17 13:00:00    18.3
2025-07-17 14:00:00    18.9
2025-07-17 15:00:00    19.3
2025-07-17 16:00:00    17.8
Name: celsius, dtype: float64

A date on its own selects the whole day, here 24 hours:

In [7]:
july_17 = oslo.loc["2025-07-17"]

print(len(july_17), "hours, warmest", july_17.max(), "| mean", round(july_17.mean(), 2))

24 hours, warmest 19.3 | mean 15.46


### Putting the labels back with reset_index

`reset_index` turns the index back into a column and numbers the rows from 0:

In [8]:
oslo.reset_index().head(3)

,time,celsius
0,2025-01-01 00:00:00,-3.5
1,2025-01-01 01:00:00,-5.3
2,2025-01-01 02:00:00,-5.3


An index with no name comes back as a column called `index`, unless `drop=True` throws it away. On a
filtered table, that column holds the old row numbers:

In [9]:
print(oslo_rows.reset_index().head(2))
print()
print(oslo_rows.reset_index(drop=True).head(2))

   index station                time  celsius
0      1    Oslo 2025-01-01 00:00:00     -3.5
1      5    Oslo 2025-01-01 01:00:00     -5.3

  station                time  celsius
0    Oslo 2025-01-01 00:00:00     -3.5
1    Oslo 2025-01-01 01:00:00     -5.3


The first table carries Oslo's original row numbers, 1 and 5, in the `index` column. The second
starts again from 0 with nothing extra. Keep the old labels when you will need to find the rows in
the original table again, and drop them otherwise.

### A station and a time together

`set_index` with a list makes a MultiIndex, one level for each column:

In [10]:
by_station_and_time = readings.set_index(["station", "time"])

print(by_station_and_time.index.names, "| sorted:", by_station_and_time.index.is_monotonic_increasing)
by_station_and_time.head(3)

['station', 'time'] | sorted: False


,,celsius
station,time,
Bergen,2025-01-01,-3.6
Oslo,2025-01-01,-3.5
Svalbard,2025-01-01,-14.6


Every reading is now labeled by two values, and `celsius` is the only column left. The index is not
sorted, because the rows still take turns between the stations. A tuple looks up one reading, and a
single station's name returns all of that station's rows, indexed by time:

In [11]:
print(by_station_and_time.loc[("Oslo", "2025-07-17 15:00")])
print()
print(by_station_and_time.loc["Svalbard"].shape)

celsius    19.3
Name: (Oslo, 2025-07-17 15:00:00), dtype: float64

(8760, 1)


### Sorting the MultiIndex, and selecting a range

`sort_index` puts the rows in order by station and then by time. A range of times within a station
needs that order, and `pd.IndexSlice` lets the tuple hold a slice, with one part for each level:

In [12]:
in_order = by_station_and_time.sort_index()
hours = pd.IndexSlice

print("sorted:", in_order.index.is_monotonic_increasing)
in_order.loc[hours["Oslo", "2025-07-17 13:00":"2025-07-17 16:00"], :]

sorted: True


celsius
station time                        
Oslo    2025-07-17 13:00:00     18.3
        2025-07-17 14:00:00     18.9
        2025-07-17 15:00:00     19.3
        2025-07-17 16:00:00     17.8

The four hours are the ones the time index gave above. Common errors shows the same line on the
unsorted table.

### Every station at a single hour with xs

`xs` takes a cross-section, which keeps one value of one level and every value of the others.

In [13]:
print("Oslo's highest of the year:", oslo.max())

in_order.xs("2025-07-17 15:00", level="time")

Oslo's highest of the year: 19.3


,celsius
station,
Bergen,19.2
Oslo,19.3
Svalbard,8.2
Tromso,16.1


The result is indexed by the level that was left, `station`, so it reads as a small table of the
four stations at that hour. Oslo's 19.3 is its peak for the year.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions
notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/09-the-index-solutions.ipynb).

**1.** Make a table of the stations indexed by name, and print Kirkenes's latitude with `loc`.

In [14]:
# your code here

**2.** Make Tromso's readings a Series indexed by time, and show its readings on 9 July from 13:00
to 17:00.

In [15]:
# your code here

**3.** From the same Series, print how many readings 9 July has and their mean, rounded to two
places.

In [16]:
# your code here

**4.** Turn Tromso's Series back into a table with `reset_index`, and show its first three rows.

In [17]:
# your code here

**5.** With a sorted station and time MultiIndex, show every station's reading at 03:00 on 8
January, using `xs`.

In [18]:
# your code here

## Common errors

### KeyError: "None of ['Station'] are in the columns"

In [19]:
readings.set_index("Station")

KeyError: "None of ['Station'] are in the columns"

`set_index` looks the name up among the columns, and column names match exactly, case included. The
message lists the names it could not find, and the column is `station`.

### ValueError: cannot insert station, already exists

In [20]:
kept = readings.set_index("station", drop=False)

kept.reset_index()

ValueError: cannot insert station, already exists

`drop=False` kept `station` as a column as well as making it the index. `reset_index` then tried to
put the index back as a column named `station`, and a table cannot have two columns with one name.
Use `reset_index(drop=True)` to discard the index, since the column is still there, or leave out
`drop=False` when you set it.

### UnsortedIndexError: 'MultiIndex slicing requires the index to be lexsorted: slicing on levels [1], lexsort depth 0'

In [21]:
by_station_and_time.loc[hours["Oslo", "2025-07-17 13:00":"2025-07-17 16:00"], :]

UnsortedIndexError: 'MultiIndex slicing requires the index to be lexsorted: slicing on levels [1], lexsort depth 0'

This is the range from the worked examples, on the table before `sort_index`. A range of labels is
"from here to there", and pandas finds it in a MultiIndex only once the rows are in order. Level 1
is `time`, the level being sliced, and a lexsort depth of 0 means the index is not sorted by any
level. Call `sort_index()` first, as the worked example did. A single lookup by tuple works on an
unsorted index, which is why the error appears only when you ask for a range.

### No error, and 8,760 rows where one value was expected

In [22]:
by_station = readings.set_index("station")
tromso = by_station.loc["Tromso", "celsius"]

print(type(tromso).__name__, len(tromso))

Series 8760


The same `loc[label, column]` that returned one latitude from the stations table returned a Series
of 8,760 readings here, because the label `Tromso` appears on 8,760 rows. Code written against the
first case, such as `round(by_station.loc["Tromso", "celsius"], 1)` meant to print one number, gets
a Series instead. When a table's labels should be unique, say so when you set them, and pandas
checks.

### ValueError: Index has duplicate keys: Index(['Bergen', 'Oslo', 'Svalbard', 'Tromso'], dtype='object', name='station')

In [23]:
print(stations.set_index("station", verify_integrity=True).shape)

readings.set_index("station", verify_integrity=True)

(5, 1)


ValueError: Index has duplicate keys: Index(['Bergen', 'Oslo', 'Svalbard', 'Tromso'], dtype='object', name='station')

`verify_integrity=True` makes `set_index` check that every label is unique and raise if any repeats,
naming the labels that do. The first line passed, because each name appears once in the stations
table, and all four station names repeat in the readings.

## Recap

- Read a table's index, and see the labels a filter keeps
- Make a column the index with `set_index`, and look rows up by label with `loc`
- Select an hour, a range of hours or a whole day from a time index
- Move the labels back into a column with `reset_index`, keeping or dropping the old ones
- Build a MultiIndex of station and time, sort it before slicing a range, and cut across it with
  `xs`

## What is next

The **Missing Data** notebook looks at Svalbard's silent day directly, at what `NaN`, `None` and
`pd.NA` are and what `dropna` and `fillna` do to them.

---

&#8592; **Previous:** [Boolean Filtering](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/08-boolean-filtering.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Missing Data](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/10-missing-data.ipynb) &#8594;
